In [0]:
%sql
create table b_sql.b_practice.entries ( 
name varchar(20),
address varchar(20),
email varchar(20),
floor int,
resources varchar(10));

insert into b_sql.b_practice.entries 
values ('A','Bangalore','A@gmail.com',1,'CPU'),('A','Bangalore','A1@gmail.com',1,'CPU'),('A','Bangalore','A2@gmail.com',2,'DESKTOP')
,('B','Bangalore','B@gmail.com',2,'DESKTOP'),('B','Bangalore','B1@gmail.com',2,'DESKTOP'),('B','Bangalore','B2@gmail.com',1,'MONITOR')

In [0]:
%sql
select * from b_sql.b_practice.entries;

In [0]:
%sql
with total_visit as (select name,count(*)as total_count,listagg(distinct resources,',') WITHIN GROUP (ORDER BY resources) as resource_used from b_sql.b_practice.entries group by name),
floor_count as (select name,floor,count(*) as total_visit from b_sql.b_practice.entries group by name,floor),
floor_rnk as(select name,floor,total_visit,rank() over(partition by name order by total_visit desc) as rnk from floor_count)
select total_visit.name,total_count as total_visit,floor_rnk.floor as most_visited_floor,resource_used from total_visit join floor_rnk on total_visit.name=floor_rnk.name and floor_rnk.rnk=1
order by name,total_visit

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
df_entries=spark.read.table("b_sql.b_practice.entries")
df_entries.display()

In [0]:


df_total = (
    df_entries
    .groupBy("name")
    .agg(
        count("*").alias("total_visit"),
        concat_ws(
            ",",
            sort_array(
                collect_set("resources"),
                asc=False
            )
        ).alias("resources")
    )
    .orderBy(col("total_visit").desc())
)

df_total.display()
df_flag=df_entries.groupBy("name","floor").agg(count("*").alias("floor_visit"))
w=Window.partitionBy(col("name")).orderBy(col("floor_visit").desc())
df_flag=df_flag.withColumn("rnk",row_number().over(w)).filter(col("rnk")==1)
df_flag.display()

df_total.alias("t").join(df_flag.alias("f"),col("t.name")==col("f.name"),"inner").drop(col("f.name"),"rnk","floor_visit").display()
